In [1]:
from dotenv import load_dotenv
load_dotenv()

import os

API_KEY = os.getenv("ALPHA_VANTAGE_API_KEY")

print(API_KEY)


2UQ6PO6AOFDYC51G


In [2]:
import requests
import pandas as pd
BASE_URL = "https://www.alphavantage.co/query"

In [3]:
def fetch_stock_data(symbol):
    
    params = {
        "function" : "TIME_SERIES_DAILY",
        "symbol": symbol,
        "apikey" : API_KEY 
    }

    response = requests.get(BASE_URL, params=params)
    print(response)
    return response.json()

In [4]:
data = fetch_stock_data("RELIANCE.BSE")
data

<Response [200]>


{'Meta Data': {'1. Information': 'Daily Prices (open, high, low, close) and Volumes',
  '2. Symbol': 'RELIANCE.BSE',
  '3. Last Refreshed': '2026-03-20',
  '4. Output Size': 'Compact',
  '5. Time Zone': 'US/Eastern'},
 'Time Series (Daily)': {'2026-03-20': {'1. open': '1395.7500',
   '2. high': '1430.0000',
   '3. low': '1395.7500',
   '4. close': '1414.5500',
   '5. volume': '1129685'},
  '2026-03-19': {'1. open': '1385.0500',
   '2. high': '1415.8000',
   '3. low': '1376.4000',
   '4. close': '1385.3500',
   '5. volume': '1739524'},
  '2026-03-18': {'1. open': '1402.9000',
   '2. high': '1412.9500',
   '3. low': '1398.7500',
   '4. close': '1408.5000',
   '5. volume': '615653'},
  '2026-03-17': {'1. open': '1400.0000',
   '2. high': '1406.0000',
   '3. low': '1388.6000',
   '4. close': '1396.4500',
   '5. volume': '688080'},
  '2026-03-16': {'1. open': '1381.9000',
   '2. high': '1397.4500',
   '3. low': '1363.4000',
   '4. close': '1395.0500',
   '5. volume': '604567'},
  '2026-03-1

In [5]:
data.keys()

dict_keys(['Meta Data', 'Time Series (Daily)'])

In [6]:
time_series = data["Time Series (Daily)"]
len(time_series)

100

In [7]:
import sys
import os

sys.path.append(os.path.abspath(".."))

from backend.api.market_data import fetch_stock_data



def pase_stock_data(data) :
    """
    Extract time serise data from API response
    """

    if "Time Series (Daily)" not in data:
        raise Exception("Time Serises data not found in API response")

    return data["Time Series (Daily)"]


data = fetch_stock_data("RELIANCE.BSE")




In [8]:
time_series = pase_stock_data(data)
print(list(time_series.keys())[:5])


['2026-03-20', '2026-03-19', '2026-03-18', '2026-03-17', '2026-03-16']


In [9]:
import pandas as pd

df = pd.DataFrame(time_series).T
df = df.rename(columns={
    "1. open" :"Open",
    "2. high" :"High" ,  
    "3. low"  :"Low" , 
    "4. close" :"Close"  ,
    "5. volume" : "Volume"
})

df = df.astype(float)
df["Volume"] = df["Volume"].astype(int)

df.index = pd.to_datetime(df.index)

df = df.sort_index(ascending=False)
df.reset_index(inplace=True)
df.rename(columns={"index":"Date"},inplace=True)
df = df.set_index("Date")
print(df[:5])

               Open     High      Low    Close   Volume
Date                                                   
2026-03-20  1395.75  1430.00  1395.75  1414.55  1129685
2026-03-19  1385.05  1415.80  1376.40  1385.35  1739524
2026-03-18  1402.90  1412.95  1398.75  1408.50   615653
2026-03-17  1400.00  1406.00  1388.60  1396.45   688080
2026-03-16  1381.90  1397.45  1363.40  1395.05   604567


In [10]:
df.info()

<class 'pandas.DataFrame'>
DatetimeIndex: 100 entries, 2026-03-20 to 2025-10-27
Data columns (total 5 columns):
 #   Column  Non-Null Count  Dtype  
---  ------  --------------  -----  
 0   Open    100 non-null    float64
 1   High    100 non-null    float64
 2   Low     100 non-null    float64
 3   Close   100 non-null    float64
 4   Volume  100 non-null    int64  
dtypes: float64(4), int64(1)
memory usage: 4.7 KB


In [11]:
describe_data = df.describe()
describe_data = describe_data.astype(float)
describe_data["Volume"] =  describe_data["Volume"].astype(int)
describe_data


,Open,High,Low,Close,Volume
count,100.000000,100.000000,100.000000,100.000000,100
mean,1477.095000,1490.183000,1464.723000,1476.773000,747810
std,67.716266,64.347758,69.034887,65.786968,620730
min,1325.000000,1353.000000,1307.000000,1345.550000,102402
25%,1419.750000,1431.800000,1400.137500,1412.637500,289542
50%,1482.275000,1492.400000,1469.375000,1480.775000,558602
75%,1539.962500,1550.925000,1531.400000,1540.825000,1078520
max,1592.500000,1611.200000,1578.000000,1592.450000,3536849


In [12]:
df.isnull().sum()

Open      0
High      0
Low       0
Close     0
Volume    0
dtype: int64

In [13]:
df.dtypes

Open      float64
High      float64
Low       float64
Close     float64
Volume      int64
dtype: object

In [14]:
df.head()

,Open,High,Low,Close,Volume
Date,,,,,
2026-03-20,1395.75,1430.00,1395.75,1414.55,1129685
2026-03-19,1385.05,1415.80,1376.40,1385.35,1739524
2026-03-18,1402.90,1412.95,1398.75,1408.50,615653
2026-03-17,1400.00,1406.00,1388.60,1396.45,688080
2026-03-16,1381.90,1397.45,1363.40,1395.05,604567


In [15]:
if df.isnull().sum().sum() > 0:
    print("Missing Values found")
else :
    print("Missing Values not found")


Missing Values not found


In [16]:
if (df["High"] < df["Low"]).any() :
    print("Invalid price data")
else :
    print("Price Data Valid")

Price Data Valid


In [17]:
def validated_data(df) :
    """
    Validated stock data
    """

    print("\n --- Data Info ---")
    print(df.info())

    print("\n --- Summary Statistics ---")
    describe_data = df.describe()
    describe_data = describe_data.astype(float).round(2)
    describe_data["Volume"] =  describe_data["Volume"].astype(int)
    print(describe_data)

    print("\n --- Missing Value found ---")
    print(df.isnull().sum())

    #Logical Checks

    if df.isnull().sum().sum() > 0 :
        raise Exception("Missing values found")
    
    print("\n Data validation successful ✅")


In [18]:
validated_data(df)


 --- Data Info ---
<class 'pandas.DataFrame'>
DatetimeIndex: 100 entries, 2026-03-20 to 2025-10-27
Data columns (total 5 columns):
 #   Column  Non-Null Count  Dtype  
---  ------  --------------  -----  
 0   Open    100 non-null    float64
 1   High    100 non-null    float64
 2   Low     100 non-null    float64
 3   Close   100 non-null    float64
 4   Volume  100 non-null    int64  
dtypes: float64(4), int64(1)
memory usage: 4.7 KB
None

 --- Summary Statistics ---
          Open     High      Low    Close   Volume
count   100.00   100.00   100.00   100.00      100
mean   1477.09  1490.18  1464.72  1476.77   747810
std      67.72    64.35    69.03    65.79   620730
min    1325.00  1353.00  1307.00  1345.55   102402
25%    1419.75  1431.80  1400.14  1412.64   289542
50%    1482.28  1492.40  1469.38  1480.78   558602
75%    1539.96  1550.93  1531.40  1540.82  1078520
max    1592.50  1611.20  1578.00  1592.45  3536849

 --- Missing Value found ---
Open      0
High      0
Low       0


In [19]:
df = df.copy()

In [20]:
df.columns.str.lower()

Index(['open', 'high', 'low', 'close', 'volume'], dtype='str')

In [21]:
df.isnull().sum()

Open      0
High      0
Low       0
Close     0
Volume    0
dtype: int64

In [22]:
df.dropna()

,Open,High,Low,Close,Volume
Date,,,,,
2026-03-20,1395.75,1430.00,1395.75,1414.55,1129685
2026-03-19,1385.05,1415.80,1376.40,1385.35,1739524
2026-03-18,1402.90,1412.95,1398.75,1408.50,615653
2026-03-17,1400.00,1406.00,1388.60,1396.45,688080
2026-03-16,1381.90,1397.45,1363.40,1395.05,604567
...,...,...,...,...,...
2025-10-31,1491.00,1497.80,1482.00,1486.50,268477
2025-10-30,1500.00,1503.25,1484.00,1488.45,772640
2025-10-29,1489.00,1508.00,1488.70,1504.05,750041


In [23]:
df["Close_norm"] = (df["Close"] /  df["Close"].max()).astype(float).round(2)

print(df)

               Open     High      Low    Close   Volume  Close_norm
Date                                                               
2026-03-20  1395.75  1430.00  1395.75  1414.55  1129685        0.89
2026-03-19  1385.05  1415.80  1376.40  1385.35  1739524        0.87
2026-03-18  1402.90  1412.95  1398.75  1408.50   615653        0.88
2026-03-17  1400.00  1406.00  1388.60  1396.45   688080        0.88
2026-03-16  1381.90  1397.45  1363.40  1395.05   604567        0.88
...             ...      ...      ...      ...      ...         ...
2025-10-31  1491.00  1497.80  1482.00  1486.50   268477        0.93
2025-10-30  1500.00  1503.25  1484.00  1488.45   772640        0.93
2025-10-29  1489.00  1508.00  1488.70  1504.05   750041        0.94
2025-10-28  1480.15  1492.10  1478.00  1487.15  1341632        0.93
2025-10-27  1461.25  1484.95  1458.70  1484.00   393023        0.93

[100 rows x 6 columns]


In [24]:
df["retuens"] = (df["Close"].pct_change()).astype(float).round(5)


In [25]:
df["pct_change"] = (df["Close"].pct_change() * 100).astype(float).round(2)
df

,Open,High,Low,Close,Volume,Close_norm,retuens,pct_change
Date,,,,,,,,
2026-03-20,1395.75,1430.00,1395.75,1414.55,1129685,0.89,NaN,NaN
2026-03-19,1385.05,1415.80,1376.40,1385.35,1739524,0.87,-0.02064,-2.06
2026-03-18,1402.90,1412.95,1398.75,1408.50,615653,0.88,0.01671,1.67
2026-03-17,1400.00,1406.00,1388.60,1396.45,688080,0.88,-0.00856,-0.86
2026-03-16,1381.90,1397.45,1363.40,1395.05,604567,0.88,-0.00100,-0.10
...,...,...,...,...,...,...,...,...
2025-10-31,1491.00,1497.80,1482.00,1486.50,268477,0.93,0.00145,0.14
2025-10-30,1500.00,1503.25,1484.00,1488.45,772640,0.93,0.00131,0.13
2025-10-29,1489.00,1508.00,1488.70,1504.05,750041,0.94,0.01048,1.05


In [26]:
stats = {}

# core return stats
stats["mean"] = round(float(df["retuens"].mean()),6)
stats["median"] = round(float(df["retuens"].median()),6)
stats["std_dev"] = round(float(df["retuens"].std()),6)
stats["variance"] =  round(float(df["retuens"].var()),6)

# Extremes

stats["min"] =  round(float(df["retuens"].min()),6)
stats["max"] =  round(float(df["retuens"].max()),6)

#Percentiles

stats["percentiles"] = {
    "25" :round(float(df["retuens"].quantile(0.25)),6),
    "50" :round(float(df["retuens"].quantile(0.50)),6),
    "75" :round(float(df["retuens"].quantile(0.75)),6),
    "5"  :round(float(df["retuens"].quantile(0.05)),6),
    "95" :round(float(df["retuens"].quantile(0.95)),6),

}

print(stats)

{'mean': 0.00056, 'median': -3e-05, 'std_dev': 0.012337, 'variance': 0.000152, 'min': -0.03317, 'max': 0.04626, 'percentiles': {'25': -0.00654, '50': -3e-05, '75': 0.0084, '5': -0.018351, '95': 0.022004}}


In [56]:
def compute_relationships(df):

    corr = df[["Open","High","Low","Close"]].corr()
    cov = df[["Open","High","Low","Close"]].cov()
    
    #round
    corr = corr.round(4)
    cov = cov.round(2)
    

    return {
        "correlation": corr.to_dict(),
        "covariance": cov.to_dict(),
        "correlation_matrix" : corr.values.tolist(),
        "lables" : corr.columns.tolist()

        
    }

In [57]:
compute_relationships(df)

{'correlation': {'Open': {'Open': 1.0,
   'High': 0.9865,
   'Low': 0.9818,
   'Close': 0.9628},
  'High': {'Open': 0.9865, 'High': 1.0, 'Low': 0.9846, 'Close': 0.9837},
  'Low': {'Open': 0.9818, 'High': 0.9846, 'Low': 1.0, 'Close': 0.9891},
  'Close': {'Open': 0.9628, 'High': 0.9837, 'Low': 0.9891, 'Close': 1.0}},
 'covariance': {'Open': {'Open': 4585.49,
   'High': 4298.41,
   'Low': 4589.93,
   'Close': 4289.3},
  'High': {'Open': 4298.41, 'High': 4140.63, 'Low': 4373.62, 'Close': 4164.44},
  'Low': {'Open': 4589.93, 'High': 4373.62, 'Low': 4765.82, 'Close': 4492.29},
  'Close': {'Open': 4289.3,
   'High': 4164.44,
   'Low': 4492.29,
   'Close': 4327.93}},
 'correlation_matrix': [[1.0, 0.9865, 0.9818, 0.9628],
  [0.9865, 1.0, 0.9846, 0.9837],
  [0.9818, 0.9846, 1.0, 0.9891],
  [0.9628, 0.9837, 0.9891, 1.0]],
 'lables': ['Open', 'High', 'Low', 'Close']}

In [ ]:
{
        "correlation": corr.to_dict(),
        "covariance": cov.to_dict(),
        "correlation_matrix" : corr.values.tolist(),
        "lables" : corr.columns.tolist()

        
    }